# SVD / SVD++ / Item-KNN Models

## 0. Imports & data loading

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Project imports
from model.SVD import SurpriseSVDModel, SurpriseSVDppEnsemble
from model.KNN import ItemKNNModel
from model.baseline import SurpriseBaselineOnlyModel
from ensemble.ensemble import RatingEnsemble
from utils.split import FrequencyBandSplitter, FrequencyBandEvaluator
from utils.predict import Predictor

In [ ]:
TRAIN_PATH = '../data/train.csv'

full_data = pd.read_csv(TRAIN_PATH)

train_data, test_data = train_test_split(full_data, test_size=0.1, random_state=42)

print(f'Train: {len(train_data):,} rows | Test: {len(test_data):,} rows')
train_data.head()

In [ ]:
# Evaluator
splitter = FrequencyBandSplitter(mode='threshold', low_threshold=1, high_threshold=10)
splitter.fit(train_data)
evaluator = FrequencyBandEvaluator(splitter)

---
## 1. SVD  (`SurpriseSVDModel`)

Wraps Surprise SVD with hyperparameters (n_factors=150, n_epochs=30, lr_all=0.005, reg_all=0.1, biased=True).

In [ ]:
svd = SurpriseSVDModel(
    n_factors=150,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.1,
    biased=True,
    clip_range=(1, 10),
    verbose=True,
)
svd.fit(train_data)

svd_predictions = svd.predict_df(test_data, round_predictions=True)
mae_svd = np.mean(np.abs(svd_predictions['prediction'] - test_data['rating']))
print(f'MAE for SVD: {mae_svd:.4f}')

evaluator.evaluate(svd, test_data)

---
## 2. Item-based KNN  (`ItemKNNModel`)

Sparse cosine-similarity KNN on the item×user matrix.

In [ ]:
item_knn = ItemKNNModel(
    k=10,
    metric='cosine',
    clip_range=(1, 10),
)
item_knn.fit(train_data)

knn_predictions = item_knn.predict_df(test_data, round_predictions=True)
mae_knn = np.mean(np.abs(knn_predictions['prediction'] - test_data['rating']))
print(f'MAE for Item KNN: {mae_knn:.4f}')

evaluator.evaluate(item_knn, test_data)

---
## 3. SVD++ Ensemble  (`SurpriseSVDppEnsemble`)

Trains 3 SVD++ models with seeds [42, 7, 2026] and averages predictions
to reduce variance.

In [ ]:
svdpp = SurpriseSVDppEnsemble(
    seeds=[42, 7, 2026],
    n_factors=150,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.1,
    biased=True,
    clip_range=(1, 10),
    verbose=True,
)
svdpp.fit(train_data)

svdpp_predictions = svdpp.predict_df(test_data, round_predictions=True)
mae_svdpp = np.mean(np.abs(svdpp_predictions['prediction'] - test_data['rating']))
print(f'MAE for SVD++ Ensemble: {mae_svdpp:.4f}')

evaluator.evaluate(svdpp, test_data)

---
## 4. Combined Ensemble

Implements ensemble (SVD 60 %, Bias 25 %, Item-KNN 15 %) using `RatingEnsemble`.

In [ ]:
# Bias baseline
baseline = SurpriseBaselineOnlyModel()
baseline.fit(train_data)

ensemble = RatingEnsemble(
    models=[svd, baseline, item_knn],
    strategy='weighted',
    weights=[0.60, 0.25, 0.15],
    clip_range=(1, 10),
)

ensemble_predictions = ensemble.predict_df(test_data, round_predictions=True)
mae_ensemble = np.mean(np.abs(ensemble_predictions['prediction'] - test_data['rating']))
print(f'MAE for SVD+Bias+KNN Ensemble: {mae_ensemble:.4f}')

evaluator.evaluate(ensemble, test_data)

### 4b. Optimise weights automatically on a validation split

In [ ]:
# Split training data into train and validation for weight optimisation
train_df, val_df = train_test_split(train_data, test_size=0.2, random_state=42)

# Use RatingEnsemble's built-in weight optimiser (minimises RMSE on validation set)
optimised_ensemble = RatingEnsemble(
    models=[svd, baseline, item_knn],
    clip_range=(1, 10),
)
optimised_weights = optimised_ensemble.fit_weights_optimized(val_df)

print('Optimised weights (fitted on validation set):', optimised_weights)

# Evaluate the optimised ensemble on the held-out test set
opt_predictions = optimised_ensemble.predict_df(test_data, round_predictions=True)
mae_opt = np.mean(np.abs(opt_predictions['prediction'] - test_data['rating']))
print(f'MAE on test set after weight optimisation: {mae_opt:.4f}')

---
## 5. Train on full data & generate submission

In [ ]:
# Re-train every model on the FULL dataset before submitting
svd_full = SurpriseSVDModel(
    n_factors=150, n_epochs=30, lr_all=0.005, reg_all=0.1,
    biased=True, clip_range=(1, 10), verbose=True,
)
svd_full.fit(full_data)

baseline_full = SurpriseBaselineOnlyModel()
baseline_full.fit(full_data)

item_knn_full = ItemKNNModel(k=10, metric='cosine', clip_range=(1, 10))
item_knn_full.fit(full_data)

final_ensemble = RatingEnsemble(
    models=[svd_full, baseline_full, item_knn_full],
    strategy='weighted',
    weights=[0.60, 0.25, 0.15],
    clip_range=(1, 10),
)

predictor = Predictor(
    test_path='../data/test.csv',
    save_path='../data/svd_knn_ensemble.csv',
    round_predictions=True,
)
predictor.predict(final_ensemble)
print('Saved submission to ../data/svd_knn_ensemble.csv')

### 5b. SVD++ submission

In [ ]:
svdpp_full = SurpriseSVDppEnsemble(
    seeds=[42, 7, 2026],
    n_factors=150, n_epochs=30, lr_all=0.005, reg_all=0.1,
    biased=True, clip_range=(1, 10), verbose=True,
)
svdpp_full.fit(full_data)

predictor_svdpp = Predictor(
    test_path='../data/test.csv',
    save_path='../data/svdpp_ensemble.csv',
    round_predictions=True,
)
predictor_svdpp.predict(svdpp_full)
print('Saved submission to ../data/svdpp_ensemble.csv')